In [2]:
import pandas as pd

df = pd.read_csv("data/vehicles.csv", low_memory=False)
print(df.shape)
print()
print(df["atvType"].value_counts(dropna=False))
print()
print(df["fuelType1"].value_counts())
print()
print(df["year"].min(), "→", df["year"].max())

(50242, 84)

atvType
NaN               43452
Hybrid             1873
EV                 1572
FFV                1538
Diesel             1238
Plug-in Hybrid      447
CNG                  50
FCV                  42
Bifuel (CNG)         20
Bifuel (LPG)          8
eFCV                  2
Name: count, dtype: int64

fuelType1
Regular Gasoline     31324
Premium Gasoline     15759
Electricity           1572
Diesel                1310
Midgrade Gasoline      173
Natural Gas             60
Hydrogen                42
Name: count, dtype: int64

1984 → 2027


In [3]:
import os
print(os.getcwd())

C:\dev\car-price-ml


In [4]:
import pandas as pd

df = pd.read_csv("data/vehicles.csv", low_memory=False)
print(df.shape)
print()
print(df["atvType"].value_counts(dropna=False))
print()
print(df["fuelType1"].value_counts())
print()
print(df["year"].min(), "→", df["year"].max())

(50242, 84)

atvType
NaN               43452
Hybrid             1873
EV                 1572
FFV                1538
Diesel             1238
Plug-in Hybrid      447
CNG                  50
FCV                  42
Bifuel (CNG)         20
Bifuel (LPG)          8
eFCV                  2
Name: count, dtype: int64

fuelType1
Regular Gasoline     31324
Premium Gasoline     15759
Electricity           1572
Diesel                1310
Midgrade Gasoline      173
Natural Gas             60
Hydrogen                42
Name: count, dtype: int64

1984 → 2027


In [5]:
cols = ["year", "make", "model", "cylinders", "displ", "drive", "trany",
        "VClass", "fuelType1", "atvType", "comb08", "combE", "range"]

print(df[cols].info())
print()
print(df["VClass"].value_counts().head(15))
print()
print(df["drive"].value_counts(dropna=False))

<class 'pandas.DataFrame'>
RangeIndex: 50242 entries, 0 to 50241
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   year       50242 non-null  int64  
 1   make       50242 non-null  str    
 2   model      50242 non-null  str    
 3   cylinders  48623 non-null  float64
 4   displ      48625 non-null  float64
 5   drive      49056 non-null  str    
 6   trany      50231 non-null  str    
 7   VClass     50242 non-null  str    
 8   fuelType1  50240 non-null  str    
 9   atvType    6790 non-null   str    
 10  comb08     50242 non-null  int64  
 11  combE      50242 non-null  float64
 12  range      50242 non-null  int64  
dtypes: float64(3), int64(3), str(7)
memory usage: 5.0 MB
None

VClass
Compact Cars                          6625
Midsize Cars                          5886
Subcompact Cars                       5822
Large Cars                            2819
Two Seaters                           2531
Standard Pickup 

In [6]:
import numpy as np

# 1. Catégorisation
def categorie(t):
    if t == "EV":
        return "electrique"
    if t == "Hybrid":
        return "hybride"
    if pd.isna(t) or t in ("FFV", "Diesel"):
        return "thermique"
    return "autre"

df["categorie"] = df["atvType"].apply(categorie)
print(df["categorie"].value_counts())

# 2. Simplifier VClass
def simplifier_classe(v):
    v = v.lower()
    if "pickup" in v:
        return "pickup"
    if "sport utility" in v or "suv" in v:
        return "suv"
    if "van" in v or "minivan" in v:
        return "van"
    if "station wagon" in v or "wagon" in v:
        return "break"
    if "two seater" in v:
        return "biplace"
    if "subcompact" in v or "minicompact" in v:
        return "citadine"
    if "compact" in v:
        return "compacte"
    if "midsize" in v:
        return "berline"
    if "large" in v:
        return "grande_berline"
    return "autre"

df["classe"] = df["VClass"].apply(simplifier_classe)
print()
print(df["classe"].value_counts())

# 3. Simplifier drive
def simplifier_drive(d):
    if pd.isna(d):
        return "inconnu"
    d = d.lower()
    if "front" in d:
        return "traction"
    if "rear" in d:
        return "propulsion"
    return "integrale"

df["transmission"] = df["drive"].apply(simplifier_drive)
print()
print(df["transmission"].value_counts())

# 4. Boîte de vitesses
df["boite"] = np.where(
    df["trany"].fillna("").str.contains("Manual"), "manuelle", "automatique"
)
print()
print(df["boite"].value_counts())

categorie
thermique     46228
hybride        1873
electrique     1572
autre           569
Name: count, dtype: int64

classe
suv               9905
citadine          7493
pickup            6973
compacte          6625
berline           5886
break             3067
grande_berline    2819
autre             2564
biplace           2531
van               2379
Name: count, dtype: int64

transmission
integrale     17354
traction      15876
propulsion    15826
inconnu        1186
Name: count, dtype: int64

boite
automatique    36945
manuelle       13297
Name: count, dtype: int64


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

th = df[df["categorie"] == "thermique"].copy()

num_cols = ["year", "cylinders", "displ"]
cat_cols = ["classe", "transmission", "boite", "fuelType1"]

X = th[num_cols + cat_cols]
y = th["comb08"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ],
    sparse_threshold=0
)

model_th = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingRegressor(random_state=42))
])

model_th.fit(X_train, y_train)
pred = model_th.predict(X_test)

baseline = np.full(len(y_test), y_train.mean())

print("Baseline MAE :", round(mean_absolute_error(y_test, baseline), 2))
print("MAE          :", round(mean_absolute_error(y_test, pred), 2))
print("R²           :", round(r2_score(y_test, pred), 3))

scores = cross_val_score(model_th, X_train, y_train, cv=5, scoring="r2")
print("CV R²        :", scores.mean().round(3), "±", scores.std().round(3))

Baseline MAE : 3.84
MAE          : 1.12
R²           : 0.906
CV R²        : 0.91 ± 0.002


In [10]:
hy = df[df["categorie"] == "hybride"].copy()

X_hy = hy[num_cols + cat_cols]
y_hy = hy["comb08"]

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_hy, y_hy, test_size=0.2, random_state=42
)

model_hy = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingRegressor(random_state=42))
])

model_hy.fit(Xh_train, yh_train)
pred_hy = model_hy.predict(Xh_test)

print("Baseline MAE :", round(mean_absolute_error(yh_test, np.full(len(yh_test), yh_train.mean())), 2))
print("MAE          :", round(mean_absolute_error(yh_test, pred_hy), 2))
print("R²           :", round(r2_score(yh_test, pred_hy), 3))

scores_hy = cross_val_score(model_hy, Xh_train, yh_train, cv=5, scoring="r2")
print("CV R²        :", scores_hy.mean().round(3), "±", scores_hy.std().round(3))


Baseline MAE : 7.99
MAE          : 1.82
R²           : 0.925
CV R²        : 0.926 ± 0.012


In [11]:
el = df[df["categorie"] == "electrique"].copy()

print(el[["comb08", "combE", "range"]].describe())
print()
print(el["classe"].value_counts())
print()
print("Valeurs manquantes :")
print(el[["cylinders", "displ", "drive", "trany"]].isnull().sum())

            comb08        combE        range
count  1572.000000  1572.000000  1572.000000
mean     93.415394    37.597868   273.824427
std      18.209855     8.546723    78.251425
min      28.000000    23.141000    29.000000
25%      80.000000    31.650025   239.000000
50%      92.000000    36.547850   281.500000
75%     106.250000    42.002650   315.000000
max     146.000000   121.000000   520.000000

classe
suv               753
grande_berline    199
berline           166
pickup            146
compacte          136
citadine           70
break              64
biplace            24
autre              10
van                 4
Name: count, dtype: int64

Valeurs manquantes :
cylinders    1572
displ        1571
drive           8
trany           9
dtype: int64


In [13]:
el = df[df["categorie"] == "electrique"].copy()

num_cols_el = ["year", "range"]
cat_cols_el = ["classe", "transmission"]

X_el = el[num_cols_el + cat_cols_el]
y_el = el["combE"]

Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    X_el, y_el, test_size=0.2, random_state=42
)

preprocessor_el = ColumnTransformer(
    [
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols_el),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_el)
    ],
    sparse_threshold=0
)

model_el = Pipeline([
    ("prep", preprocessor_el),
    ("model", HistGradientBoostingRegressor(random_state=42))
])

model_el.fit(Xe_train, ye_train)
pred_el = model_el.predict(Xe_test)

print("Baseline MAE :", round(mean_absolute_error(ye_test, np.full(len(ye_test), ye_train.mean())), 2))
print("MAE          :", round(mean_absolute_error(ye_test, pred_el), 2))
print("R²           :", round(r2_score(ye_test, pred_el), 3))

scores_el = cross_val_score(model_el, Xe_train, ye_train, cv=5, scoring="r2")
print("CV R²        :", scores_el.mean().round(3), "±", scores_el.std().round(3))

Baseline MAE : 5.8
MAE          : 3.6
R²           : 0.542
CV R²        : 0.502 ± 0.098


In [14]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(model_th, "models/model_thermique.pkl")
joblib.dump(model_hy, "models/model_hybride.pkl")
joblib.dump(model_el, "models/model_electrique.pkl")

print("3 modèles sauvegardés")
print([f for f in os.listdir("models")])

3 modèles sauvegardés
['model_electrique.pkl', 'model_hybride.pkl', 'model_thermique.pkl']
